This notebook is primarly for applying analytics and Business logics, the cleaning or processing is performed in the 02_processed.ipynb

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
CLEANED_DIRECTORY =  r"A:\JnJ_assignment_data_science\Data\Silver"
STORAGE_DIRECTORY = r"A:\JnJ_assignment_data_science\Data\Gold"

In [3]:
employees_df=pd.read_csv(CLEANED_DIRECTORY + r"\FE_cleaned_table.csv")


We dont require all the columns from the Silver processed dataframe

In [4]:
filtered_cols = [
    "EMP_ID",
    "Location",
    "Possible_nearest_station",
    "Distance_to_cover(kms)",
    "distance_to_station(kms)",
    "station_to_workplace(kms)",
    "total_time"
]
employees_df = employees_df[filtered_cols]


This function is essential to find the possibility for commuting

We have 5 main conditions:

- if the distance is less than 1.5 kms -> it is okay to walk

- if the distance is less than 4kms -> considering the environment. There is more possibility to go with bike rather than public transport

- if the distance is greater than 4 kms time taken is between 30 mins to 45 minutes then it is Excellent to commute regularly to and fro

- if the time taken is between 45 mins to 60 minutes then it is Good to commute regularly to and fro

- if the time taken is between 60 mins to 80 minutes then it is Time consuming to commute regularly to and fro (but 1 hour to and fro is comparable)

- the final condition is more like an outlier, making ot the office travelling (*2 hours to and 2 hours fro is nearly an tiring task*)

In [5]:
def analysis_commuting(total_time,distance):
    if distance <= 1.4:
        return "Walk to work"
    elif distance <=4:
        return "Bike friendly"
    elif distance > 4 and 30 <=total_time <= 45:
        return "Excellent"
    elif 45 <= total_time <= 60:
        return "Good"
    elif 60 <= total_time <=80:
        return "Time consuming"
    else:
        return "Too far to commute regularly"


employees_df["commute_group"] = employees_df.apply(lambda df : analysis_commuting(df["total_time"],df["Distance_to_cover(kms)"]),axis=1)

This function takes the reference from the commuting method analysis and breaks down the metric for the adoption of Deutschland ticket

I have grouped as follows

- walk to work (**very less chance to take deutschland ticket**)
- bike friendly (**the chances are equally less but when it snows or rains the chances could eb to take a public trasport**)
- Excellent and Good (**These groups will completely depend on the public transport more often**)

In [6]:
def adoption_potential(commute_group):
    if commute_group == "Walk to work":
        return "Very Low"
    elif commute_group == "Bike friendly":
        return "Low"
    elif commute_group == "Excellent":
        return "Very High"
    elif commute_group == "Good":
        return "Very High"
    elif commute_group == "Time consuming":
        return "High"
    else:
        return "Low"


employees_df["Deutschlandticket_Metric"] = employees_df["commute_group"].apply(adoption_potential)

Uptill here we make up the Gold serving layer for the employees table

In [7]:
employees_df.to_csv(STORAGE_DIRECTORY + r"\Gold_Employees.csv",index=False)

This part focuses on summary of the various sectors

- no of employees for the **commute groups** with the percentage distribution
- no of employees with high chances for deustchland ticket adoption with the percentage distribution

In [8]:
def grouping_summary(dataframe):
    summary = (dataframe["commute_group"].value_counts().rename_axis("commute_groups").reset_index(name="Employees"))
    summary["Percentage"] = round((summary["Employees"] / len(dataframe)) * 100,1)
    return summary

grouping_summary(employees_df)


,commute_groups,Employees,Percentage
0,Good,271,54.4
1,Excellent,86,17.3
2,Time consuming,78,15.7
3,Bike friendly,39,7.8
4,Too far to commute regularly,13,2.6
5,Walk to work,11,2.2


In [9]:
def adoption_potential_summary(dataframe):
    summary = (dataframe["Deutschlandticket_Metric"].value_counts().rename_axis("Potential").reset_index(name="Employees"))
    summary["Percentage"] = round((summary["Employees"] / len(dataframe)) * 100,1)
    return summary

adoption_potential_summary(employees_df)


,Potential,Employees,Percentage
0,Very High,357,71.7
1,High,78,15.7
2,Low,52,10.4
3,Very Low,11,2.2


This function is very important for analysis

- This shows the distribution of employees and relevant details with respect to the location they come from

    - we do a groupby based on the following
    
        - count of **EMP_ID**
        - mean of **total_time** (tells us the avergae time to commute)
        - median of **total time** (tell us the spread across the timeline distribution)
        - mean of **home -> station** (tells us the average distance a person covers coming from that particular location)
        - mean of **home -> office** (tells us the average distance a person covers coming from that particular location)
        - sum of *(very high and high users)* (tells us the total counts of the employees who highly depend on the deutschland ticket)
        - count of possible nearest station
    
We also add % for adoption by calculating the no of deutscland ticket users with the total number of employees count. (*gives us a perspective of how the percetnge distirbution is present for the high adoption users with respect to the location*)

In [10]:
def create_area_summary(dataframe):
    
    area_summary = (dataframe.groupby("Location").agg(
            Employees=("EMP_ID", "count"),
            Avg_commute_time=("total_time", "mean"),
            Median_commute_time=("total_time", "median"),
            Avg_distance_from_home_to_station=("distance_to_station(kms)", "mean"),
            Avg_distance_from_home_to_office=("Distance_to_cover(kms)", "mean"),
            Possible_deutschland_ticket_users=("Deutschlandticket_Metric",lambda x: x.isin(["Very High", "High"]).sum()),
            Unique_stations=("Possible_nearest_station", "nunique")).reset_index()
            )
    
    area_summary["High_adoption_percentage"] = (area_summary["Possible_deutschland_ticket_users"]/area_summary["Employees"]*100)
    area_summary = area_summary.round(2)
    
    return area_summary


area_summary = create_area_summary(employees_df)

We perform a groupby by 
- splitting based on location and taking the **mode (highest occuring value)** of the nearest station and we keep it as the main station

Finally we perform a merge between the calculated table for the statistisc with the main station(*left join* why because we want all the matching values of the main station to be mapped with the *parent table **area summary***)

In [11]:
def get_main_station(dataframe):
    
    main_station = (dataframe.groupby("Location")["Possible_nearest_station"].agg(lambda x: x.mode()[0]).reset_index()
                    .rename(columns={"Possible_nearest_station": "Main_station"}))
    return main_station

main_station = get_main_station(employees_df)
area_summary = area_summary.merge(main_station,on="Location",how="left")

We have 3 unique functions to calculate the connectivity score helps with strong connectivity and weak connectivity points

- function station_access_score makes use of 

    - it checks the distance for walk or by bus and rates based on the distance 
        -if the distance is less then the score is good (*time saved*) and more the distance lower the score (*time lost*)


- function commute_time_score makes use of 

    - it checks the time for travelling and rates based on the time taken for travelling 
        -if the avg time taken is less then the score is good (*less time to travel*) and more the time taken lower the score (*too time consuming*)


- function adoption_score makes use of 

    - it checks desutchland ticket adoption percentage of the employees for that particular location 
        -if the adoption score% is high then the score is good  and lower the adoption score% lower the score


- function station_diversity_score makes use of 

    - it checks the unique station and employee can take coming from that location 
        -if the count of unique stations are high then the score is good  and lower the count of unique stations lower the score


In [12]:
def station_score(distance):
    
    if distance <= 1:
        return 4
    elif distance <= 2:
        return 3
    elif distance <= 3:
        return 2
    else:
        return 1
    
def commute_time_score(time):
    
    if time <= 40:
        return 4
    elif time <= 50:
        return 3
    elif time <= 60:
        return 2
    else:
        return 1
    

def adoption_score(percentage):
    
    if percentage >= 80:
        return 4
    elif percentage >= 60:
        return 3
    elif percentage >= 40:
        return 2
    else:
        return 1
    
    
def unique_station_score(station_count):
    
    if station_count >= 4:
        return 4
    elif station_count == 3:
        return 3
    elif station_count == 2:
        return 2
    else:
        return 1


In [13]:
area_summary["Station_access_score"] = area_summary["Avg_distance_from_home_to_station"].apply(station_score)

area_summary["Commute_time_score"] = area_summary["Avg_commute_time"].apply(commute_time_score)

area_summary["Adoption_score"] = area_summary["High_adoption_percentage"].apply(adoption_score)

area_summary["Station_diversity_score"] = area_summary["Unique_stations"].apply(unique_station_score)

This section is for the connectivity score (**very important factor**)

- We simply take the sum of the scores which we found earlier in the section
- for advancedment we also give a *weighted sum for multiplication*. This way we have the power to give importance for the respectie scores

    - Commute_time_score is given 40% importance
    - Station_access_score is given 30% importance
    - Adoption_score is given 20% importance
    - Station_diversity_score is given 10% importance

In [14]:
area_summary["Connectivity_score"] = (
    area_summary["Commute_time_score"] * 0.40
    + area_summary["Station_access_score"] * 0.30
    + area_summary["Adoption_score"] * 0.20
    + area_summary["Station_diversity_score"] * 0.10
)
area_summary["Connectivity_score"] = area_summary["Connectivity_score"].round(2)

Based on the score we construct the
- Strong connectivity 
- weak connectivity

In [15]:
def connectivity_for_transport(score):

    if score >= 3.0:
        return "Strong public transport connectivity"
    else:
        return "Less attractive public transport"
    
area_summary["Connectivity_category"] = area_summary["Connectivity_score"].apply(connectivity_for_transport)

In [16]:
strong_connectivity_areas = area_summary[
    area_summary["Connectivity_category"] == "Strong public transport connectivity"
]

In [17]:
less_attractive_areas = area_summary[
    area_summary["Connectivity_category"] == "Less attractive public transport"
]

Final visualization layer table is present

In [18]:
final_area_summary = area_summary[
    [
        "Location",
        "Employees",
        "Avg_commute_time",
        "Avg_distance_from_home_to_station",
        "High_adoption_percentage",
        "Unique_stations",
        "Main_station",
        "Connectivity_score",
        "Connectivity_category"
    ]
]


In [19]:
final_area_summary.to_csv(STORAGE_DIRECTORY + r"\Gold_Area_Summary.csv",index=False)